# M1 · 데이터 수집과 재현 가능한 원본 보관

**입력:** M0에서 확인한 ②~⑤ 승인 API, 선택 자료 ⑥, 이미 받은 ①·⑦ 파일. **출력:** API 전체 행의 변경 불가능한 JSON·CSV 스냅샷, 수집 로그, 다운로드 파일과의 비교 보고서. M0에서 해시를 확인한 ①·⑦은 다시 내려받지 않는다.

같은 응답을 다시 받으면 SHA-256 기반 파일명을 재사용하므로 중복 스냅샷이 생기지 않는다. API 키는 `.env`에서만 읽고 출력·저장하지 않는다. 실행할 때마다 현재 API 내용이 바뀌었는지 확인한다.

## 1. M0 확인 결과와 수집 설정

In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import zipfile
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
import pandas as pd
import requests
from dotenv import dotenv_values
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists()), None)
assert ROOT is not None, 'VS Code에서 SANUP-P 프로젝트 폴더를 열고 이 노트북을 실행하세요.'
RAW = ROOT / 'data' / 'personal' / 'raw'
REPORTS = ROOT / 'data' / 'personal' / 'reports'
REPORTS.mkdir(parents=True, exist_ok=True)
SECRETS = dotenv_values(ROOT / '.env') if (ROOT / '.env').exists() else {}
print('프로젝트:', ROOT)
print('원본 폴더:', RAW)

In [ ]:
SOURCES = [
    {'id': '01', 'dataset_id': '15140383', 'role': 'SIF 사례 분석·RAG 후보',
     'file': 'source_01_sif_archive.xlsx', 'encoding': None, 'key_env': None, 'uuid': None},
    {'id': '02', 'dataset_id': '15084672', 'role': '사고재해자 대시보드',
     'file': 'source_02_accident_injured.csv', 'encoding': 'cp949',
     'key_env': 'SOURCE_02_ACCIDENT_INJURED_API_KEY', 'uuid': 'uddi:8a4523d3-816b-4eb0-ad2f-9d88d5895e71'},
    {'id': '03', 'dataset_id': '15084674', 'role': '사고사망자 대시보드',
     'file': 'source_03_accident_deaths.csv', 'encoding': 'cp949',
     'key_env': 'SOURCE_03_ACCIDENT_DEATHS_API_KEY', 'uuid': 'uddi:6b3e32f7-6431-4a2f-978a-e207b058ae1d'},
    {'id': '04', 'dataset_id': '15064487', 'role': '사업장 수 대시보드',
     'file': 'source_04_establishments.csv', 'encoding': 'cp949',
     'key_env': 'SOURCE_04_ESTABLISHMENTS_API_KEY', 'uuid': 'uddi:d8b9a405-5ec5-445f-b1d1-99484da54212'},
    {'id': '05', 'dataset_id': '15064491', 'role': '사망만인율 대시보드',
     'file': 'source_05_fatality_rate.csv', 'encoding': 'cp949',
     'key_env': 'SOURCE_05_FATALITY_RATE_API_KEY', 'uuid': 'uddi:041d50b2-e50a-4bdd-95d9-bae6bdf4cee2'},
    {'id': '06', 'dataset_id': '15162988', 'role': 'RAG 원문 위치 목록',
     'file': 'source_06_rag_source_index.csv', 'encoding': 'utf-8-sig',
     'key_env': 'SOURCE_06_RAG_SOURCE_INDEX_API_KEY', 'uuid': 'uddi:6aad2c14-2c8e-4381-bc3b-9397ffe99a03'},
    {'id': '07', 'dataset_id': '15161288', 'role': 'M7 용어사전',
     'file': 'source_07_safety_terms.zip', 'encoding': None, 'key_env': None, 'uuid': None},
]
for source in SOURCES:
    source['page_url'] = f"https://www.data.go.kr/data/{source['dataset_id']}/fileData.do"
    source['api_url'] = (f"https://api.odcloud.kr/api/{source['dataset_id']}/v1/{source['uuid']}"
                         if source['uuid'] else None)

m0_path = REPORTS / 'm0_source_validation.json'
assert m0_path.exists(), 'M0 노트북을 먼저 실행하세요.'
m0_report = json.loads(m0_path.read_text(encoding='utf-8'))
required_ids = {'02', '03', '04', '05', '06'}
required_checks = [r for r in m0_report['api_checks'] if r['source_id'] in required_ids]
assert len(required_checks) == len(required_ids), 'M0 API 점검 기록이 빠졌습니다. M0 노트북을 먼저 실행하세요.'
print('M0 최신 상태:', {r['source_id']: r['status'] for r in required_checks})
print('실패한 API가 있어도 성공한 자료는 각각 수집·기록하고, 전체 완료 여부는 마지막에 따로 판정합니다.')

## 2. 페이지를 끝까지 요청하고 전체 건수를 확인

In [ ]:
def collect_pages(session, source, key, per_page=100, max_pages=100):
    rows = []
    total = None
    pages = 0
    for page in range(1, max_pages + 1):
        response = session.get(source['api_url'],
                               params={'page': page, 'perPage': per_page, 'serviceKey': key},
                               timeout=30)
        # 본문·URL에 키가 포함될 수 있으므로 에러 전문은 표시하지 않는다.
        if not response.ok:
            raise RuntimeError(f"source_{source['id']}: HTTP {response.status_code}")
        try:
            payload = response.json()
        except ValueError as exc:
            raise RuntimeError(f"source_{source['id']}: JSON 형식 오류") from None
        if not isinstance(payload, dict) or not isinstance(payload.get('data'), list):
            raise RuntimeError(f"source_{source['id']}: data 목록 누락")
        batch = payload['data']
        if not all(isinstance(row, dict) for row in batch):
            raise RuntimeError(f"source_{source['id']}: 행 형식 오류")
        reported_total = payload.get('totalCount')
        if not isinstance(reported_total, int) or reported_total < 0:
            raise RuntimeError(f"source_{source['id']}: totalCount 오류")
        if total is None:
            total = reported_total
        elif total != reported_total:
            raise RuntimeError(f"source_{source['id']}: 수집 중 totalCount 변경")
        rows.extend(batch)
        pages = page
        if len(rows) >= total:
            break
        if not batch:
            raise RuntimeError(f"source_{source['id']}: 데이터 없는 페이지에서 조기 종료")
    if len(rows) != total:
        raise RuntimeError(f"source_{source['id']}: 기대 {total}행, 실제 {len(rows)}행")
    return rows, total, pages

session = requests.Session()
collected = []
failures = []
for source in SOURCES:
    if not source['api_url']:
        continue
    key = (SECRETS.get(source['key_env']) or '').strip()
    if not key:
        print(f"source_{source['id']}: 키 미설정으로 건너뜀")
        continue
    try:
        rows, total, pages = collect_pages(session, source, key)
        collected.append({'source': source, 'rows': rows, 'total': total, 'pages': pages})
        print(f"source_{source['id']}: {total}행, {pages}페이지 수집")
    except (requests.RequestException, RuntimeError) as exc:
        # requests 예외에는 요청 URL이 들어갈 수 있어 전문 대신 유형만 출력한다.
        response = getattr(exc, 'response', None)
        status_code = getattr(response, 'status_code', None)
        # requests 예외 전문에는 키가 포함된 URL이 들어갈 수 있어 기록하지 않는다.
        controlled_message = str(exc) if isinstance(exc, RuntimeError) else ''
        if status_code is None and controlled_message.startswith(f"source_{source['id']}: HTTP "):
            try:
                status_code = int(controlled_message.rsplit(' ', 1)[-1])
            except ValueError:
                status_code = None
        failure = {'source_id': source['id'], 'error_type': type(exc).__name__,
                   'http_status': status_code, 'detail': controlled_message or None}
        failures.append(failure)
        print(f"source_{source['id']}: 수집 실패 ({controlled_message or type(exc).__name__})")

## 3. SHA-256 스냅샷 저장 · 다운로드 파일과 행 내용 비교

In [ ]:
from collections import Counter
from decimal import Decimal, InvalidOperation

def canonical(value):
    if pd.isna(value):
        return ''
    try:
        number = Decimal(str(value))
        if number.is_finite():
            return format(number.normalize(), 'f')
    except InvalidOperation:
        pass
    return str(value)

def row_multiset(frame, columns):
    return Counter(tuple(canonical(row[col]) for col in columns)
                   for _, row in frame.iterrows())

summaries = []
for item in collected:
    s, rows = item['source'], item['rows']
    api_frame = pd.DataFrame(rows)
    portal_file = RAW / f"source_{s['id']}" / s['file']
    portal_frame = pd.read_csv(portal_file, encoding=s['encoding'])
    same_columns = set(api_frame.columns) == set(portal_frame.columns)
    same_rows = (same_columns and len(api_frame) == len(portal_frame) and
                 row_multiset(api_frame, sorted(api_frame.columns)) ==
                 row_multiset(portal_frame, sorted(api_frame.columns)))
    snapshot = {'source_id': s['id'], 'dataset_id': s['dataset_id'],
                'api_url': s['api_url'], 'total_count': item['total'],
                'per_page': 100, 'records': rows}
    content = (json.dumps(snapshot, ensure_ascii=False, sort_keys=True, separators=(',', ':')) + '\n').encode('utf-8')
    sha256 = hashlib.sha256(content).hexdigest().upper()
    api_dir = RAW / f"source_{s['id']}" / 'api'
    api_dir.mkdir(parents=True, exist_ok=True)
    json_path = api_dir / f"source_{s['id']}_api_{sha256[:12]}.json"
    csv_path = api_dir / f"source_{s['id']}_api_{sha256[:12]}.csv"
    if json_path.exists():
        assert hashlib.sha256(json_path.read_bytes()).hexdigest().upper() == sha256, '스냅샷 이름 충돌'
    else:
        json_path.write_bytes(content)
    csv_bytes = api_frame.to_csv(index=False).encode('utf-8-sig')
    if csv_path.exists():
        assert csv_path.read_bytes() == csv_bytes, 'API CSV 이름 충돌'
    else:
        csv_path.write_bytes(csv_bytes)
    summaries.append({'source_id': s['id'], 'dataset_id': s['dataset_id'],
                      'collected_at': datetime.now(ZoneInfo('Asia/Seoul')).isoformat(timespec='seconds'),
                      'rows': len(rows), 'pages': item['pages'], 'total_count': item['total'],
                      'sha256': sha256, 'json_path': str(json_path.relative_to(ROOT)),
                      'csv_path': str(csv_path.relative_to(ROOT)), 'portal_rows': len(portal_frame),
                      'columns_match': bool(same_columns), 'row_content_match': bool(same_rows)})

display(pd.DataFrame([{k: s[k] for k in ('source_id', 'rows', 'pages', 'portal_rows',
                                          'columns_match', 'row_content_match')}
                      for s in summaries]))

## 4. 로그와 수집 보고서 저장

In [ ]:
log_path = RAW / 'api_collection_log.csv'
fields = ['source_id', 'dataset_id', 'collected_at', 'rows', 'pages', 'total_count',
          'sha256', 'json_path', 'csv_path', 'portal_rows', 'columns_match', 'row_content_match']
existing = []
if log_path.exists():
    with log_path.open(encoding='utf-8-sig', newline='') as stream:
        existing = list(csv.DictReader(stream))
seen = {(r['source_id'], r['sha256']) for r in existing}
new_rows = [s for s in summaries if (s['source_id'], s['sha256']) not in seen]
if new_rows:
    with log_path.open('a', encoding='utf-8-sig', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        if not existing:
            writer.writeheader()
        writer.writerows(new_rows)

collected_ids = {s['source_id'] for s in summaries}
required_summaries = [s for s in summaries if s['source_id'] in required_ids]
missing_required = sorted(required_ids - collected_ids)
required_complete = (not missing_required and len(required_summaries) == len(required_ids)
                     and all(s['columns_match'] and s['row_content_match']
                             and s['rows'] == s['total_count'] == s['portal_rows']
                             for s in required_summaries))
report = {'collected_at': datetime.now(ZoneInfo('Asia/Seoul')).isoformat(timespec='seconds'),
          'api_sources': summaries, 'failures': failures,
          'required_source_ids': sorted(required_ids),
          'missing_required_sources': missing_required,
          'file_only_sources': ['01', '07', '08'],
          'required_complete': required_complete}
report_path = REPORTS / 'm1_collection_summary.json'
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print('새 로그 행:', len(new_rows), '전체 로그 행:', len(existing) + len(new_rows))
print('수집 기록:', report_path)
print('필수 ②~⑥ 전체 수집·파일 대조 완료:', report['required_complete'])
if not report['required_complete']:
    print('미완료 API:', report['missing_required_sources'], '오류 기록:', failures)

## ⑥·⑧ 파일 수집 기록 확인

In [ ]:
from pathlib import Path
import hashlib, json, zipfile
import pandas as pd
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists()), None)
assert ROOT is not None
raw = ROOT / 'data' / 'personal' / 'raw'
guide_dir = raw / 'source_08'
guide_index = pd.read_csv(guide_dir / 'kosha_guide_index.csv')
expected = {f'{row.guide_id}.pdf': str(row.sha256).lower() for row in guide_index.itertuples(index=False)}
seen = set()
archive_checks = []
zip_checks = []
for archive_path in (guide_dir / 'kosha_1.zip', guide_dir / 'kosha_2.zip'):
    with zipfile.ZipFile(archive_path) as archive:
        pdf_members = [name for name in archive.namelist() if name.lower().endswith('.pdf')]
        zip_checks.append({'file': archive_path.name, 'bytes': archive_path.stat().st_size,
                           'sha256': hashlib.sha256(archive_path.read_bytes()).hexdigest(),
                           'pdf_count': len(pdf_members)})
        for member in pdf_members:
            name = Path(member).name
            assert name in expected and name not in seen, f'색인 누락 또는 ZIP 중복: {name}'
            seen.add(name)
            archive_bytes = archive.read(member)
            extracted_bytes = (guide_dir / 'pdf' / name).read_bytes()
            archive_hash = hashlib.sha256(archive_bytes).hexdigest().lower()
            extracted_hash = hashlib.sha256(extracted_bytes).hexdigest().lower()
            archive_checks.append({'file': name, 'archive_matches_index': archive_hash == expected[name],
                                   'extracted_matches_archive': archive_hash == extracted_hash})
assert seen == set(expected) and len(expected) == 18
assert all(row['archive_matches_index'] and row['extracted_matches_archive'] for row in archive_checks)
report_dir = raw / 'source_06' / 'reports'
report_index = pd.read_csv(report_dir / 'manifest.csv')
report_checks = []
for row in report_index.itertuples(index=False):
    pdf = report_dir / row.filename
    digest = hashlib.sha256(pdf.read_bytes()).hexdigest() if pdf.is_file() else None
    report_checks.append({'doc_id': row.doc_id, 'sha256_ok': digest is not None and digest.upper() == str(row.sha256).upper()})
assert len(report_checks) == 3 and all(row['sha256_ok'] for row in report_checks)
summary = {'source_06_report_pdfs': len(report_index), 'source_08_guide_pdfs': len(guide_index),
           'source_08_zip_archives': zip_checks, 'source_08_pdf_pages': int(guide_index.pdf_pages.sum()),
           'archive_pdf_checks': archive_checks, 'guide_sha256_mismatches': 0,
           'report_sha256_mismatches': 0}
target = ROOT / 'data' / 'personal' / 'reports' / 'm1_rag_files_summary.json'
target.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print('⑥ 보고서 PDF 3건: 매니페스트 해시 일치')
print('⑧ ZIP 2개 / PDF 18개 / 549쪽: ZIP 안 원본·압축 해제본·색인 해시 일치')


## M1 마무리 기록 — 2026-10-03

### 이번 단계에서 한 일

②~⑥ API를 페이지 끝까지 요청하고, 행 수·열 구조·CSV 원본과의 행 내용을 대조한다. API가 다시 같은 결과를 주면 콘텐츠 해시가 같은 파일을 재사용하고, 바뀐 응답은 다른 SHA-256 이름으로 보존한다. API 키와 키가 들어간 요청 URL은 보고서·로그에 저장하지 않는다.

### 확인된 수집 결과

- 이전 정상 실행(2026-10-01)에서는 ②~⑤ 각 30행, ⑥ 5행을 API에서 받아 한 페이지씩 끝까지 읽었다. `totalCount`, 열 목록, 다운로드 CSV의 모든 행을 대조했고 그 실행에서는 모두 일치했다.
- 2026-10-03 재검증에서는 ②~⑥ 전체 API를 끝까지 수집했다. ②~⑤는 각 30행, ⑥은 5행이며 모든 API의 totalCount, 열 구성, 다운로드 CSV의 전체 행 내용이 일치했다. ③도 이번 실행에서 정상 수집·대조했다.
- ⑥ 보고서 PDF 3개와 ⑧ GUIDE PDF 18개는 파일 해시를 확인했다. GUIDE는 두 ZIP 내부 파일의 해시, 압축 해제본, 인덱스의 SHA-256이 서로 일치하고 전체 549쪽이다.
- 과거 수집 로그에는 첫 비교에서 ⑤의 행 대조가 실패했던 이력이 있다. 그 뒤 10월 1일 실행 결과와 이번 ⑤ 재호출에서는 수치 표기를 정규화해 30행 전체가 일치했다. 이전 로그는 감사 이력이라 지우지 않는다.

### 다음 단계에서 사용할 것

정제·분석 단계는 다운로드 파일과 API 스냅샷을 구분해 출처를 연결한다. RAG 자료는 ⑥의 링크 목록을 실제 사고 본문처럼 넣지 않고, 해시가 확인된 보고서 PDF와 GUIDE 원본·페이지 메타데이터를 사용한다.

### 알게 된 점과 주의할 점

- `totalCount`까지 도달했다는 것만으로 다운로드 파일과 같은 내용이라고 할 수 없다. 열 이름과 각 행의 값을 따로 비교해야 한다.
- ⑤처럼 숫자 `0.30`과 `0.3`은 문자열 그대로 비교하면 다른 값처럼 보인다. 비교할 때 숫자를 정규화하되, 원본 파일의 표기 자체는 바꾸지 않는다.
- 실패한 API를 조용히 건너뛰면 오래된 파일과 새 응답이 섞여도 완료로 오해할 수 있다. 이번 보고서는 실패 API와 누락 목록을 남기며, 필수 자료 모두가 최신 호출·파일 대조를 통과하지 않으면 M1 완료로 표시하지 않는다.
- 재실행할 때 ①·⑦을 다시 내려받거나 원본 CSV를 덮어쓰지 않는다. API 스냅샷은 별도 하위 폴더에 저장한다.
- ZIP 해시를 수집 시점에 따로 기록하지 않은 기존 자료는 현재 ZIP 내부 PDF와 추출본 일치만 검증할 수 있다. 이번 실행부터는 ZIP 자체의 SHA-256도 함께 기록한다.

### M1 완료 여부

2026-10-03 기준 ②~⑥ 필수 API를 모두 수집했고, 5개 자료의 열 구성과 전체 행 내용이 포털 원본 CSV와 일치했다. 실패·누락 자료가 없어 M1 수집 및 원본 대조를 완료했다.
